# 관련성 검증을 추가한 RAG (Human-in-the-loop)

기본 RAG 는 검색된 문서를 무조건 답변 근거로 쓴다. 하지만 질문이 모호하거나 검색 결과가 엉뚱하면 답변 품질이 떨어진다. 이 노트북은 두 가지 안전장치를 더한다:

1. **문서 관련성 평가(grade)** — 검색된 문서가 질문과 정말 관련 있는지 LLM 이 yes/no 로 판정
2. **Human-in-the-loop(`interrupt`)** — 관련 없으면 그래프를 멈추고 **사용자에게 질문을 다시** 받음

```
START → agent ──(검색 필요?)──▶ retrieve → grade ──(관련?)──▶ generate → END
          └─(불필요)─▶ END                        └─(무관)─▶ rewrite(사람에게 재질문) → agent
```

> `OPENAI_API_KEY` 필요. retriever 는 01 노트북과 같은 공개 문서로 구성한다.

## 환경 변수 준비

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ.setdefault("USER_AGENT", "ai-agent-study")
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 가 .env 에 없습니다"
print("환경변수 로드 완료")

## 0. Retriever 준비

[basics 복습] 01 노트북과 동일하게 공개 웹 문서를 로드·청킹·임베딩해 retriever 를 만들고, 그걸 **도구(tool)** 로 등록한다. (검색 파이프라인 자체는 01 참고)

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.tools.retriever import create_retriever_tool

pages = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/").load()
docs = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(pages)
vectorstore = Chroma.from_documents(documents=docs, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

retriever_tool = create_retriever_tool(
    retriever,
    "retrieve_agent_docs",
    "Search and return information about LLM-based AI agents (planning, memory, tool use).",
)
tools = [retriever_tool]
print("retriever 도구 준비 완료")

## Step 0. Graph State

[basics 복습] `MessagesState` 를 상속한다. 재작성된 질문을 따로 누적하기 위해 `questions` 필드(역시 `add_messages` 리듀서)를 추가한다.

In [ ]:
from typing import Annotated, Literal, Sequence
from langchain_core.messages import BaseMessage
from langgraph.graph import MessagesState
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI

class State(MessagesState):
    questions: Annotated[Sequence[BaseMessage], add_messages]

llm = ChatOpenAI(model="gpt-4o", temperature=0)

## Step 1. agent 노드 — 검색 도구 호출 여부 판단
[basics 복습] `bind_tools` 한 LLM 이 검색이 필요하면 tool_calls 를, 아니면 일반 답변을 만든다.

In [ ]:
def agent(state: State):
    """질문을 보고 retriever 도구를 호출할지, 그냥 끝낼지 결정한다."""
    print("##### AGENT #####")
    llm_with_tools = llm.bind_tools(tools)
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

## Step 2. 문서 관련성 평가

[basics 복습] 구조화 출력(`with_structured_output`)으로 LLM 이 **yes/no** 만 내게 한다. 이 함수는 조건부 엣지의 라우터로 쓰여 관련 있으면 `generate`, 없으면 `rewrite` 로 분기한다.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

class Grade(BaseModel):
    """관련성 평가 이진 점수"""
    binary_score: str = Field(description="관련성 점수 'yes' 또는 'no'")

def grade_documents(state: State) -> Literal["generate", "rewrite"]:
    """검색된 문서가 질문과 관련 있는지 평가한다."""
    print("##### CHECK RELEVANCE #####")
    grader = llm.with_structured_output(Grade)
    grader_prompt = ChatPromptTemplate.from_template(
        """You are a grader assessing relevance of a retrieved document to a user question.
        Retrieved document:\n{context}\n
        User question: {question}
        If the question is too short/vague to determine intent, grade as not relevant and return 'no'.
        Give a binary score 'yes' or 'no'."""
    )
    chain = grader_prompt | grader

    # 재작성된 질문이 있으면 그걸, 없으면 최초 질문 사용
    question = state["questions"][-1].content if state["questions"] else state["messages"][0].content
    docs = state["messages"][-1].content   # retrieve 결과(ToolMessage)

    score = chain.invoke({"question": question, "context": docs}).binary_score
    if score == "yes":
        print("---DECISION: RELEVANT---")
        return "generate"
    print("---DECISION: NOT RELEVANT---")
    return "rewrite"

## Step 3. Human-in-the-loop — 사용자에게 질문 다시 받기

**`interrupt`** 는 그래프를 그 지점에서 **일시정지** 하고 사용자 입력을 기다린다. 관련 문서를 못 찾았을 때, 더 명확한 질문을 사람에게 직접 받아 다시 시도한다.

> `interrupt` 가 동작하려면 그래프 컴파일 시 **checkpointer 가 필수** 다.

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.types import interrupt

def rewrite(state: State):
    """검색이 부실하면 사용자에게 더 명확한 질문을 다시 받는다 (Human-in-the-loop)."""
    print("\n🤖 검색 결과 개선을 위해 더 명확한 질문을 입력해주세요.")
    feedback = interrupt("New Question:")   # 여기서 그래프가 멈추고 사용자 입력 대기
    return {"messages": [feedback], "questions": [HumanMessage(content=feedback)]}

## Step 4. generate 노드 — 최종 답변 생성
[basics 복습] 검색된 문서를 근거로 RAG 프롬프트로 답한다 (프롬프트 인라인).

In [ ]:
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are an assistant for question-answering. Use the retrieved context to answer. "
     "If you don't know, say so. Three sentences max.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

def generate(state: State):
    print("##### GENERATE #####")
    question = state["questions"][-1].content if state["questions"] else state["messages"][0].content
    docs = state["messages"][-1].content
    response = llm.invoke(RAG_PROMPT.format_messages(context=docs, question=question))
    return {"messages": [response]}

## Step 5. 그래프 조립

```
START → agent ──tools_condition──▶ retrieve → grade_documents ──▶ generate → END
          └─ END                                   └──▶ rewrite → agent (다시)
```

In [ ]:
from langgraph.graph import END, StateGraph, START
from langgraph.prebuilt import ToolNode, tools_condition

graph_builder = StateGraph(State)
graph_builder.add_node("agent", agent)
graph_builder.add_node("retrieve", ToolNode([retriever_tool]))
graph_builder.add_node("rewrite", rewrite)
graph_builder.add_node("generate", generate)

graph_builder.add_edge(START, "agent")
# 검색 필요하면 retrieve, 아니면 END
graph_builder.add_conditional_edges("agent", tools_condition, {"tools": "retrieve", END: END})
# 검색 후 관련성 평가 → generate or rewrite
graph_builder.add_conditional_edges("retrieve", grade_documents)
graph_builder.add_edge("generate", END)
graph_builder.add_edge("rewrite", "agent")   # 재질문 후 다시 agent 로

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# interrupt 사용을 위해 checkpointer 필수
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

## 테스트 1. 검색이 불필요한 인사말 — 바로 종료

In [ ]:
config = {"configurable": {"thread_id": "1"}}
for event in graph.stream(
    {"messages": [("user", "안녕하세요! 오늘 기분이 좋네요.")]},
    config, stream_mode="values",
):
    if "messages" in event:
        event["messages"][-1].pretty_print()

## 테스트 2. 모호한 질문 → 관련성 'no' → interrupt 발생

"최신 에이전트" 처럼 모호하면 검색 결과가 부실해 `rewrite` 로 가고 그래프가 멈춘다.

In [ ]:
config = {"configurable": {"thread_id": "2"}}
for event in graph.stream(
    {"messages": [("user", "agent")]},
    config, stream_mode="values",
):
    if "messages" in event:
        event["messages"][-1].pretty_print()

# 그래프가 rewrite 에서 멈췄는지 확인 (다음 실행 노드)
print("\n다음 실행 노드(중단 지점):", graph.get_state(config).next)

## 테스트 3. `Command(resume=...)` 로 사람이 더 명확한 질문 제공 → 재개

**`Command(resume=값)`** 으로 멈춘 `interrupt` 에 값을 넣어 그래프 실행을 이어간다.

In [ ]:
from langgraph.types import Command

for event in graph.stream(
    Command(resume="What is the planning component of an LLM agent?"),
    config, stream_mode="values",
):
    if "messages" in event:
        event["messages"][-1].pretty_print()

## 부록) Human-in-the-loop 기본 예제

**Human-in-the-loop** = 자동화 흐름의 핵심 지점에 사람의 판단/검증/수정을 끼워넣는 것. LLM 이 가끔 틀리므로 중요한 결정에 유용하다.
- **`interrupt(값)`**: 노드에서 그래프를 멈추고, 값을 사용자에게 보여주며 입력을 기다림
- **`Command(resume=값)`**: 멈춘 지점에 값을 주입하며 실행 재개

최소 예제로 메커니즘만 확인한다 (checkpointer 필수).

In [ ]:
import uuid
from typing import Optional
from typing_extensions import TypedDict
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START
from langgraph.types import interrupt, Command

class MiniState(TypedDict):
    foo: str
    human_value: Optional[str]

def node(state: MiniState):
    answer = interrupt("what is your age?")   # 사용자에게 보낼 질문
    print(f"> interrupt 로 받은 입력: {answer}")
    return {"human_value": answer}

mini = StateGraph(MiniState)
mini.add_node("node", node)
mini.add_edge(START, "node")
mini_graph = mini.compile(checkpointer=MemorySaver())

cfg = {"configurable": {"thread_id": str(uuid.uuid4())}}
for chunk in mini_graph.stream({"foo": "abc"}, cfg):
    print(chunk)   # __interrupt__ 가 찍히며 멈춤

In [ ]:
# 사람이 값을 주입하며 재개
for chunk in mini_graph.stream(Command(resume="30"), cfg):
    print(chunk)

## 정리

- **관련성 평가**: 검색 문서가 질문과 맞는지 `grade_documents` 가 yes/no 판정 → 분기
- **Human-in-the-loop**: 관련 없으면 `interrupt` 로 멈춰 사용자에게 재질문, `Command(resume=...)` 로 재개
- `interrupt` 사용 시 **checkpointer 필수**
- 이렇게 "검색 품질" 을 한 번 거르면 엉뚱한 근거로 답하는 일이 준다

다음: 사람 대신 **답변의 환각 여부를 LLM 이 평가** 하고, 부족하면 질문을 자동 재작성/웹검색하는 RAG.